# Visualize Model Predictions

This notebook loads a trained model and visualizes its predictions on a new sample.

In [ ]:
import sys
import json
import random
import os
import shutil
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path


project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))


from plot_utilities import *
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess
import time
from IPython.display import display
import plotly.express as px
import phate
import tempfile

# ERA5_Graph_Generator.generate_normed_sample() draws the random subset with
# np.random.choice, so the sampled graph -- and every embedding derived from it below --
# changes run-to-run unless the global RNGs are seeded first. The in-notebook methods
# pass random_state=42 explicitly, and the UMAP/concord runner scripts hardcode their own
# seed, so seeding here is what makes the sampled input itself reproducible.
SEED = 42


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()

In [ ]:
def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / id
    with open(model_path, 'r') as f:
        config = json.load(f)
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    return model
    

In [ ]:
# choose data
model_type = 'regular'


# ---------------------------------------------------------------------------
from utilities_era5 import ERA5_Graph_Generator as Data_Generator



# Paths
trained_models_path = project_root / 'ERA5' / 'trained_models' / model_type
training_logs_path = trained_models_path / 'logs'
data_path = project_root / 'ERA5' / 'data'

print('Available models:\n')
files = sorted(os.listdir(trained_models_path), reverse=True)

for i, f in enumerate(files):
    if (Path(f).suffix == ".pth") and ('adamw' in f):
        print(f'[{i}] {f}')


## 2. Load Model and Configuration

In [ ]:
# Set model ID
model_id = 'regular_papermodel2_2026-05-04_t12-01-43'



# ---------------------------------------------------------------------------
if model_id.endswith('.pth'):
    model_id = model_id[:-4]

# Load full configuration (includes model, dataset, and training configs)
config_path = f'{training_logs_path}/configs/{model_id}.json'
with open(config_path, 'r') as f:
    config = json.load(f)

# Load loss records
loss_df = pd.read_csv(f'{training_logs_path}/loss_records/{model_id}.csv')

print('Configuration:')
for key, value in config.items():
    print(f'  {key}: {value}')

print(f'\Training completed {len(loss_df)} epochs')
print(f'Final training loss: {loss_df["loss"].iloc[-1]:.4f}')
print(f'Final validation loss: {loss_df["val_loss"].iloc[-1]:.4f}')

In [ ]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]


In [ ]:
model = KernelNN(
    width=config['width'],
    ker_width=config['ker_width'],
    depth=config['depth'],
    ker_in=config['edge_attr'],
    in_width=config['node_features'],
    out_width=config['out_width']
).to(device)

# Load trained weights
model_path = f'{trained_models_path}/{model_id}.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print(f'Model loaded: {model_id}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## 3. Create New Sample

Use the same functionality as dataset creation to generate a single sample.

In [ ]:
date = "2019-06-15"

#bbox = lon_min, lat_min, lon_max, lat_max
bbox = -12, 30, 33, 75
#Generator = Data_Generator_old(data_path)
Generator = Data_Generator(data_path)
Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')


# Re-seed immediately before sampling: generate_normed_sample() consumes global np.random
# state, so anything executed between the imports cell and here would otherwise shift the
# sampled point set.
set_seed()
N=5000
print('generate data')
graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode)
sample = graph.to(device)

locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})

In [ ]:
print('predict results')
output = model(sample).detach().cpu().numpy()
X = sample.x.detach().cpu().numpy()

---
Plot

In [ ]:
res_model = pd.DataFrame(
    {
        "x1": output[:, 0],
        "x2": output[:, 1],
        "x3": output[:, 2],
    }
)
res_model = pd.concat([locations, res_model], axis=1)

figsize = (3, 3)
fig = rgb(res_model, figsize=figsize)
#fig = rgb_scale(res_model, percentile=1, figsize=figsize)

In [ ]:
def scatter(X, c):

    plt.rcParams.update({
        "font.family": "serif",                 # Use serif fonts
        "font.serif": ["Times New Roman"],      # Set to Times New Roman
        "font.size": 11,          # Base font size
        "axes.labelsize": 11,     # Axis labels
        "legend.fontsize": 10,    # Slightly smaller legend
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.loc": 'upper right',
        # Set font size in points
    })

    width = 0.45
    ratio = 1.61803398875  # Golden ratio

    linewidth = 3.31314 *1.1* 2.54 # inch
    width_in = width * linewidth  # = 5.85 inches
    height_in = width_in / ratio # Choose a suitable aspect ratio, e.g., 3:2 or 4:3

    fig, ax = plt.subplots(figsize=(width_in, height_in))

    sc = plt.scatter(X[:,0], X[:,1], c=c, cmap="viridis", s=1)
    plt.colorbar(sc, label="latitude")
    return fig, ax 


In [ ]:
os.makedirs('figures/scatter', exist_ok=True)

In [ ]:
# apply PCA to reduce to 3 dimensions
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(output)

plot, ax = scatter(X_reduced, c=locations.lat)
plt.show()
plot.savefig(f"figures/scatter/nofe_{date}_n{N}.pdf", bbox_inches="tight")

## PCA

In [ ]:
# apply PCA to reduce to 3 dimensions
pca = PCA(n_components=2)
X_reduced_pca = pca.fit_transform(X)

plot, ax = scatter(X_reduced_pca, c=locations.lat)
plt.show()
plot.savefig(f"figures/scatter/pca_{date}_n{N}.pdf", bbox_inches="tight")

## t-SNE

In [ ]:
pred_tsne = TSNE(
        n_components=3,
        perplexity=30,
        learning_rate="auto",
        random_state=42
    ).fit_transform(X)
# apply PCA to reduce to 3 dimensions
pca = PCA(n_components=2)
X_reduced_tsne = pca.fit_transform(pred_tsne)


plot, ax = scatter(X_reduced_tsne, c=locations.lat)
plt.show()
plot.savefig(f"figures/scatter/tsne_{date}_n{N}.pdf", bbox_inches="tight")

## UMAP

In [ ]:
umap_id = 'umap_buffer'
dir = f'{project_root}/ERA5/umap'
np.save(f'{dir}/{umap_id}.npy', X)
subprocess.run(f"cd {dir} && conda run -n umap python execute_umap.py {umap_id}", shell=True)
pred_umap = np.load(f'{dir}/res_{umap_id}.npy')

pca = PCA(n_components=2)
X_reduced_umap = pca.fit_transform(pred_umap)

plot, ax = scatter(X_reduced_umap, c=locations.lat)
plt.show()
plot.savefig(f"figures/scatter/umap_{date}_n{N}.pdf", bbox_inches="tight")


## PHATE

In [ ]:
pred_phate = phate.PHATE(
    n_components=3, knn=15, mds_solver='smacof', random_state=42, verbose=False
).fit_transform(X)

pca = PCA(n_components=2)
X_reduced_phate = pca.fit_transform(pred_phate)


plot, ax = scatter(X_reduced_phate, c=locations.lat)
plt.show()
plot.savefig(f"figures/scatter/phate_{date}_n{N}.pdf", bbox_inches="tight")


## Concord

In [ ]:
concord_id = 'concord_buffer'
dir = f'{project_root}/ERA5/concord'
np.save(f'{dir}/{concord_id}.npy', X)
subprocess.run(f"cd {dir} && conda run -n concord python execute_concord.py {concord_id}", shell=True)
pred_concord = np.load(f'{dir}/res_{concord_id}.npy')

pca = PCA(n_components=2)
X_reduced_concord = pca.fit_transform(pred_concord)


plot, ax = scatter(X_reduced_concord, c=locations.lat)
plt.show()
plot.savefig(f"figures/scatter/concord_{date}_n{N}.pdf", bbox_inches="tight")


## SpatialPCA

In [ ]:
spatialpca_id = 'spatialpca_buffer'
spatialpca_script = f'{project_root}/ERA5/spatialpca/execute_spatialpca_single.R'
spatialpca_coords = sample.normed_coords.detach().cpu().numpy()

# R searches the *user* library (~/Library/R/<arch>/<ver>/library) before the conda env's
# own library. CRAN-built packages there link the CRAN R framework rather than conda's
# libR, and SpatialPCA imports Rcpp -- whose foreign .so then segfaults at dyn.load.
# Pointing R_LIBS_USER at a nonexistent path lets the env's own packages win.
r_env = dict(os.environ, R_LIBS_USER='/nonexistent')

with tempfile.TemporaryDirectory() as spdir:
    np.savetxt(f'{spdir}/{spatialpca_id}_X.csv', X, delimiter=',')
    np.savetxt(f'{spdir}/{spatialpca_id}_coords.csv', spatialpca_coords, delimiter=',')
    subprocess.run(
        f'conda run -n spatialpca Rscript {spatialpca_script} {spatialpca_id} 3',
        shell=True, cwd=spdir, check=True, env=r_env,
    )
    pred_spatialpca = np.loadtxt(f'{spdir}/res_{spatialpca_id}.csv', delimiter=',', skiprows=1)

pca = PCA(n_components=2)
X_reduced_spatialpca = pca.fit_transform(pred_spatialpca)


plot, ax = scatter(X_reduced_spatialpca, c=locations.lat)
plt.show()
plot.savefig(f"figures/scatter/spatialpca_{date}_n{N}.pdf", bbox_inches="tight")


In [ ]:
# ---------------------------------------------------------------------------
# Combined 2x4 raster of all seven embeddings, sharing a single latitude colorbar.
# Sized for \includegraphics[width=\textwidth]{...} with neurips_2024.sty:
#   \textwidth = 5.5in -> page width is exactly 5.5in
#   text is 8pt (common for dense multi-panel figures; body text is 10pt)
#   saved WITHOUT bbox_inches="tight" -> nothing is cropped or rescaled
#   pdf.fonttype=42 embeds TrueType so the point sizes survive in the PDF
TEXTWIDTH_IN = 5.5   # \textwidth in neurips_2024.sty
BASE_PT = 8          # panel labels / colorbar label
TICK_PT = 7          # colorbar tick labels
GAP_IN = 0.05        # space between neighbouring panels
CBAR_W_IN = 0.055    # colorbar width
PT2IN = 1 / 72

panels = [
    ("(a)", "NOFE", X_reduced),
    ("(b)", "PCA", X_reduced_pca),
    ("(c)", "t-SNE", X_reduced_tsne),
    ("(d)", "UMAP", X_reduced_umap),
    ("(e)", "PHATE", X_reduced_phate),
    ("(f)", "CONCORD", X_reduced_concord),
    ("(g)", "SpatialPCA", X_reduced_spatialpca),
]

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": BASE_PT,
    "axes.labelsize": BASE_PT,
    "xtick.labelsize": TICK_PT,
    "ytick.labelsize": TICK_PT,
    "axes.linewidth": 0.4,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

nrows, ncols = 2, 4


def build(reserve_in):
    """Draw the raster leaving `reserve_in` inches free at the right edge.

    Returns (fig, colorbar, need_w) where `need_w` is the measured footprint of
    the colorbar plus its tick labels and rotated axis label.
    """
    panel_w = (TEXTWIDTH_IN - reserve_in - (ncols - 1) * GAP_IN) / ncols
    fig_h = nrows * panel_w + (nrows - 1) * GAP_IN

    fig, axes = plt.subplots(nrows, ncols, figsize=(TEXTWIDTH_IN, fig_h))
    fig.subplots_adjust(left=0.0, bottom=0.0, top=1.0,
                        right=(TEXTWIDTH_IN - reserve_in) / TEXTWIDTH_IN,
                        wspace=GAP_IN / panel_w, hspace=GAP_IN / panel_w)

    sc = None
    for ax, (label, name, emb) in zip(axes.ravel(), panels):
        sc = ax.scatter(emb[:, 0], emb[:, 1], c=locations.lat, cmap="viridis",
                        s=0.15, linewidths=0, rasterized=True)
        # Plain box: no ticks, no tick labels, no axis labels.
        ax.set_xticks([])
        ax.set_yticks([])
        ax.tick_params(length=0)
        ax.margins(0.06)                      # keep points off the frame
        for spine in ax.spines.values():
            spine.set_linewidth(0.4)
        ax.text(0.035, 0.965, f"{label} {name}", transform=ax.transAxes,
                ha="left", va="top", fontsize=BASE_PT,
                bbox=dict(boxstyle="square,pad=0.08", facecolor="white",
                          edgecolor="none", alpha=0.65))

    # Eighth slot stays empty.
    for ax in axes.ravel()[len(panels):]:
        ax.set_visible(False)

    # Single shared colorbar in the reserved strip, temporarily full height.
    cax = fig.add_axes([(TEXTWIDTH_IN - reserve_in + GAP_IN) / TEXTWIDTH_IN, 0.0,
                        CBAR_W_IN / TEXTWIDTH_IN, 1.0])
    cbar = fig.colorbar(sc, cax=cax)
    cbar.set_label("latitude", fontsize=BASE_PT)
    cbar.ax.tick_params(labelsize=TICK_PT, width=0.4, length=1.5, pad=1.5)
    cbar.outline.set_linewidth(0.4)

    fig.canvas.draw()
    ren = fig.canvas.get_renderer()
    tick_w = max(t.get_window_extent(ren).width for t in cbar.ax.get_yticklabels()) / fig.dpi
    label_w = cbar.ax.yaxis.label.get_window_extent(ren).height / fig.dpi  # rotated
    need_w = CBAR_W_IN + tick_w + label_w + 4 * 3 * PT2IN
    return fig, cbar, need_w


# Pass 1 measures the colorbar footprint; pass 2 lays out with exactly that room,
# so the page is exactly \textwidth wide and no glyph is clipped.
reserve = 1.0
for _ in range(2):
    fig, cbar, need_w = build(reserve)
    if reserve > need_w:
        plt.close(fig)
        reserve = need_w
    else:
        break

# Inset the colorbar vertically until its outermost tick label fits on the page.
cbar_left = TEXTWIDTH_IN - reserve + GAP_IN
cax = fig.axes[-1]
fig_h = fig.get_size_inches()[1]
inset_lo = inset_hi = 0.0
for _ in range(2):
    cax.set_position([cbar_left / TEXTWIDTH_IN, inset_lo / fig_h,
                      CBAR_W_IN / TEXTWIDTH_IN, (fig_h - inset_lo - inset_hi) / fig_h])
    fig.canvas.draw()
    ren = fig.canvas.get_renderer()
    pos = cax.get_position()
    over_lo = max(0.0, pos.y0 * fig_h - min(t.get_window_extent(ren).y0
                                            for t in cbar.ax.get_yticklabels()) / fig.dpi)
    over_hi = max(0.0, max(t.get_window_extent(ren).y1
                           for t in cbar.ax.get_yticklabels()) / fig.dpi - pos.y1 * fig_h)
    if over_lo == 0 and over_hi == 0:
        break
    inset_lo += over_lo + 1.5 * PT2IN
    inset_hi += over_hi + 1.5 * PT2IN

plt.show()
# No bbox_inches="tight": the PDF page must stay exactly \textwidth wide.
fig.savefig(f"figures/scatter/combined_{date}_n{N}.pdf")
print(f"saved {TEXTWIDTH_IN}in x {fig.get_size_inches()[1]:.2f}in | "
      f"text {BASE_PT}pt / {TICK_PT}pt, reserve {reserve:.2f}in")